# 02 — Supervised Regression
Предварительные требования + Task 1.1–1.5 на KDD Cup 1999 (`corrected`).

## Preliminary requirement (обоснование)
- Выбор датасета: KDD Cup 1999 (доступен локально и подходит для IDS).
- Пропуски: median для числовых, most_frequent для категориальных.
- Кодирование: OneHotEncoder для `protocol_type`, `service`, `flag`.
- Масштабирование: StandardScaler для числовых признаков.
- Split: stratify по attack/benign.
- Без leakage: preprocessor fit только на train и сохраняется до transform test.

In [3]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from utils.config import load_config

cfg = load_config('config.yaml')
SEED = cfg['seed']
random.seed(SEED)
np.random.seed(SEED)

FEATURES = cfg['dataset']['feature_columns']
CATS = cfg['dataset']['categorical_features']
COLS = FEATURES + ['label']

DOS = {'back.', 'land.', 'neptune.', 'pod.', 'smurf.', 'teardrop.', 'apache2.', 'udpstorm.', 'processtable.', 'mailbomb.'}
PROBE = {'ipsweep.', 'nmap.', 'portsweep.', 'satan.', 'mscan.', 'saint.'}
R2L = {'ftp_write.', 'guess_passwd.', 'imap.', 'multihop.', 'phf.', 'spy.', 'warezclient.', 'warezmaster.', 'snmpgetattack.', 'snmpguess.', 'xlock.', 'xsnoop.', 'worm.', 'named.'}
U2R = {'buffer_overflow.', 'loadmodule.', 'perl.', 'rootkit.', 'httptunnel.', 'ps.', 'sqlattack.', 'xterm.'}
SEVERITY = {'benign': 0.0, 'probe': 45.0, 'dos': 65.0, 'r2l': 82.0, 'u2r': 95.0, 'unknown': 70.0}

def family(label: str) -> str:
    if label == 'normal.':
        return 'benign'
    if label in DOS:
        return 'dos'
    if label in PROBE:
        return 'probe'
    if label in R2L:
        return 'r2l'
    if label in U2R:
        return 'u2r'
    return 'unknown'

def metrics(y_true, y_pred):
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)),
    }

df = pd.read_csv(cfg['paths']['data'], names=COLS)
df['family'] = df['label'].map(family)
df['severity_score'] = df['family'].map(SEVERITY).astype(float)
df['is_attack'] = (df['label'] != 'normal.').astype(int)

X = df[FEATURES].copy()
y = df['severity_score'].copy()

X_train, X_test, y_train, y_test, strat_train, strat_test = train_test_split(
    X, y, df['is_attack'], test_size=cfg['supervised']['test_size'],
    stratify=df['is_attack'], random_state=SEED
)

num_cols = [c for c in FEATURES if c not in CATS]

for col in num_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').astype('float32')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').astype('float32')

prep = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=True))]), CATS),
])

prep.fit(X_train)
Path(cfg['paths']['models_dir']).mkdir(parents=True, exist_ok=True)
joblib.dump(prep, cfg['paths']['preprocessor'])

X_train_t = prep.transform(X_train)
X_test_t = prep.transform(X_test)

print('Train/Test attack ratio:', round(float(strat_train.mean()), 4), round(float(strat_test.mean()), 4))
print('Shapes:', X_train_t.shape, X_test_t.shape)
print('Matrix type:', type(X_train_t))

Train/Test attack ratio: 0.8052 0.8052
Shapes: (248823, 117) (62206, 117)
Matrix type: <class 'numpy.ndarray'>


In [ ]:
# Task 1.1 + 1.2 + 1.3 (memory-safe setup)
lin = LinearRegression().fit(X_train_t, y_train)
lin_m = metrics(y_test, lin.predict(X_test_t))

rf = RandomForestRegressor(
    n_estimators=120,
    max_depth=16,
    random_state=SEED,
    n_jobs=18
).fit(X_train_t, y_train)
rf_m = metrics(y_test, rf.predict(X_test_t))

param_dist = {
    'n_estimators': [100, 150, 220],
    'max_depth': [12, 16, 24, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.7],
}

# Tune on subset to avoid OOM, then refit best params on full train
if X_train_t.shape[0] > 120000:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(X_train_t.shape[0], size=120000, replace=False)
    X_tune = X_train_t[idx]
    y_tune = y_train.iloc[idx]
else:
    X_tune = X_train_t
    y_tune = y_train

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=1),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=SEED,
    n_jobs=18,
    verbose=1,
)
search.fit(X_tune, y_tune)

best_model = RandomForestRegressor(
    **search.best_params_,
    random_state=SEED,
    n_jobs=18
)
best_model.fit(X_train_t, y_train)
best_m = metrics(y_test, best_model.predict(X_test_t))

joblib.dump(best_model, cfg['paths']['regression_model'])

pd.DataFrame([
    {'Model': 'LinearRegression', **lin_m},
    {'Model': 'RandomForest', **rf_m},
    {'Model': 'TunedRandomForest', **best_m},
]).sort_values('RMSE')

Fitting 5 folds for each of 8 candidates, totalling 40 fits


KeyboardInterrupt: 

In [ ]:
# Task 1.4
feat_names = prep.get_feature_names_out()
imp = pd.DataFrame({'feature': feat_names, 'importance': best_model.feature_importances_}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(imp['feature'][::-1], imp['importance'][::-1])
plt.title('Top-10 Feature Importance (Tuned RF)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Best params:', search.best_params_)
print('Best CV RMSE:', round(-search.best_score_, 4))
imp

## Аналитика
- **1.1:** линейная модель — базовый ориентир, но для сетевых данных обычно ограничена линейным допущением.
- **1.2:** улучшение RF над Linear указывает на нелинейные зависимости и взаимодействия признаков.
- **1.3:** выбран `RandomizedSearchCV` (5-fold), т.к. он эффективнее полного grid при широкой сетке параметров.
- **1.4:** top-features отражают интенсивность трафика и error-rate паттерны, что полезно для SOC-приоритезации.
- **1.5:** инференс реализован в `predict.py` (валидация + обработка ошибок + логирование в `utils/logger.py`).